# 02 English Dataset Transfer + City-Swap Pilot

This notebook turns the English Kaggle dataset into a usable next step, without immediately launching a full retraining loop.

Recommended order:
1. load and sanity-check the English dataset,
2. map raw English categories into the project's 9 target classes,
3. build an English location-aware evaluation slice,
4. generate English city-swap counterfactuals,
5. optionally run already-trained local models on the English slice if checkpoints are available.


## Why this is the next move

`city swap` is worth doing on English data, but only after we confirm the dataset can be projected into the same 9-class label space.

So the practical plan is:
- do **label mapping first**,
- create an **English counterfactual benchmark** next,
- only then evaluate existing models,
- retrain only if transfer actually looks weak.


In [ ]:
import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import kagglehub
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Install kagglehub first: pip install kagglehub"
    ) from exc

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ModuleNotFoundError:
    HAS_MPL = False

CWD = Path.cwd()
NOTEBOOK_DIR = CWD if (CWD / "results").exists() else (CWD / "notebooks")
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR.parent
OUTPUT_DIR = REPO_ROOT / "notebooks" / "results" / "english_transfer_pilot"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)


In [ ]:
KAGGLE_HANDLE = "rayyankauchali0/resume-dataset"
path = Path(kagglehub.dataset_download(KAGGLE_HANDLE))
print("Downloaded to:", path)

def load_jsonl(path_obj: Path):
    rows = []
    with path_obj.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def load_dataset(download_dir: Path) -> pd.DataFrame:
    candidates = list(download_dir.rglob("*.csv")) + list(download_dir.rglob("*.jsonl")) + list(download_dir.rglob("*.json"))
    if not candidates:
        zip_candidates = list(download_dir.rglob("*.zip"))
        for archive in zip_candidates:
            with zipfile.ZipFile(archive) as zf:
                extract_dir = archive.with_suffix("")
                zf.extractall(extract_dir)
        candidates = list(download_dir.rglob("*.csv")) + list(download_dir.rglob("*.jsonl")) + list(download_dir.rglob("*.json"))

    if not candidates:
        raise FileNotFoundError(f"No CSV/JSONL/JSON files found under {download_dir}")

    ranked = sorted(candidates, key=lambda p: (p.suffix != ".csv", len(str(p))))
    data_path = ranked[0]
    print("Using data file:", data_path)

    if data_path.suffix == ".csv":
        return pd.read_csv(data_path)
    if data_path.suffix == ".jsonl":
        return load_jsonl(data_path)
    return pd.read_json(data_path)

df = load_dataset(path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(3)


In [ ]:
TEXT_CANDIDATES = ["Resume_Text", "resume_text", "Resume", "Text", "Summary", "Experience", "Education", "Skills"]
CATEGORY_CANDIDATES = ["Category", "category", "Job_Role", "job_role"]
LOCATION_CANDIDATES = ["Location", "location", "City", "city", "Address", "address"]

def first_present(columns, candidates):
    for col in candidates:
        if col in columns:
            return col
    return None

text_col = first_present(df.columns, TEXT_CANDIDATES)
category_col = first_present(df.columns, CATEGORY_CANDIDATES)
location_col = first_present(df.columns, LOCATION_CANDIDATES)

if text_col is None:
    raise ValueError(f"Could not find a resume text column among: {TEXT_CANDIDATES}")

df = df.copy()
df["resume_text_work"] = df[text_col].fillna("").astype(str)
df["text_len_chars"] = df["resume_text_work"].str.len()
df["text_len_words"] = df["resume_text_work"].str.split().str.len()

print("text_col:", text_col)
print("category_col:", category_col)
print("location_col:", location_col)
df[[c for c in [text_col, category_col, location_col, "text_len_words"] if c is not None]].head(5)


In [ ]:
TARGET_9_CLASSES = [
    "backend_general_dev",
    "web_frontend",
    "sysadmin_devops_network",
    "project_product",
    "sales_account",
    "tech_support_helpdesk",
    "it_governance_leadership",
    "technical_specialized",
    "generic_it_ops",
]

CATEGORY_TO_TARGET = {
    "Data Science": "technical_specialized",
    "Database": "technical_specialized",
    "DevOps Engineer": "sysadmin_devops_network",
    "DotNet Developer": "backend_general_dev",
    "ETL Developer": "backend_general_dev",
    "Hadoop": "technical_specialized",
    "HR": "generic_it_ops",
    "Advocate": "sales_account",
    "Arts": "generic_it_ops",
    "Automation Testing": "technical_specialized",
    "Blockchain": "technical_specialized",
    "Business Analyst": "project_product",
    "Civil Engineer": "generic_it_ops",
    "Designer": "web_frontend",
    "Electrical Engineering": "generic_it_ops",
    "Health and fitness": "generic_it_ops",
    "Java Developer": "backend_general_dev",
    "Mechanical Engineer": "generic_it_ops",
    "Network Security Engineer": "sysadmin_devops_network",
    "Operations Manager": "it_governance_leadership",
    "PMO": "project_product",
    "Python Developer": "backend_general_dev",
    "SAP Developer": "technical_specialized",
    "Sales": "sales_account",
    "Testing": "tech_support_helpdesk",
    "Web Designing": "web_frontend",
}

def normalize_category(value):
    if pd.isna(value):
        return None
    key = str(value).strip()
    if key in CATEGORY_TO_TARGET:
        return CATEGORY_TO_TARGET[key]

    lowered = key.lower()
    if any(token in lowered for token in ["frontend", "web design", "ui", "ux"]):
        return "web_frontend"
    if any(token in lowered for token in ["backend", "python", "java", "dotnet", "developer", "software", "etl"]):
        return "backend_general_dev"
    if any(token in lowered for token in ["devops", "sysadmin", "network", "security", "cloud", "infrastructure"]):
        return "sysadmin_devops_network"
    if any(token in lowered for token in ["product", "project", "business analyst", "pmo"]):
        return "project_product"
    if any(token in lowered for token in ["sales", "account", "advocate", "marketing"]):
        return "sales_account"
    if any(token in lowered for token in ["support", "helpdesk", "testing", "qa"]):
        return "tech_support_helpdesk"
    if any(token in lowered for token in ["manager", "head", "director", "leadership", "operations manager"]):
        return "it_governance_leadership"
    if any(token in lowered for token in ["data", "database", "sap", "blockchain", "hadoop", "ml", "ai"]):
        return "technical_specialized"
    return "generic_it_ops"

if category_col is not None:
    df["target_9_class"] = df[category_col].apply(normalize_category)
else:
    df["target_9_class"] = "generic_it_ops"

mapping_review = (
    df.groupby([category_col, "target_9_class"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    if category_col is not None
    else pd.DataFrame({"target_9_class": df["target_9_class"].value_counts().index, "count": df["target_9_class"].value_counts().values})
)
mapping_review.to_csv(OUTPUT_DIR / "02_english_category_mapping_review.csv", index=False)
mapping_review.head(30)


In [ ]:
CITY_PATTERN = re.compile(
    r"\b("
    r"new york|san francisco|los angeles|seattle|austin|chicago|boston|atlanta|dallas|houston|"
    r"london|berlin|paris|madrid|toronto|vancouver|sydney|melbourne|singapore|dubai|"
    r"bangalore|bengaluru|mumbai|delhi|hyderabad|pune"
    r")\b",
    flags=re.IGNORECASE,
)

def extract_city_mentions(text: str) -> list[str]:
    if not text:
        return []
    return sorted({m.group(0).lower() for m in CITY_PATTERN.finditer(text)})

df["city_mentions"] = df["resume_text_work"].apply(extract_city_mentions)
df["n_city_mentions"] = df["city_mentions"].apply(len)
df["has_city_mention"] = df["n_city_mentions"] > 0

location_summary = pd.DataFrame(
    {
        "rows_total": [len(df)],
        "rows_with_city_mentions": [int(df["has_city_mention"].sum())],
        "share_with_city_mentions": [float(df["has_city_mention"].mean()) if len(df) else 0.0],
        "median_words": [float(df["text_len_words"].median()) if len(df) else 0.0],
    }
)
location_summary.to_csv(OUTPUT_DIR / "02_english_location_summary.csv", index=False)

city_counts = (
    df.explode("city_mentions")
    .dropna(subset=["city_mentions"])
    .groupby("city_mentions")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
city_counts.to_csv(OUTPUT_DIR / "02_english_city_counts.csv", index=False)

location_summary


In [ ]:
SWAP_CITIES = [
    "new york",
    "san francisco",
    "london",
    "berlin",
    "singapore",
]

def swap_cities_in_text(text: str, target_city: str) -> str:
    if not isinstance(text, str) or not text:
        return text

    def repl(match):
        original = match.group(0)
        if original.isupper():
            return target_city.upper()
        if original[:1].isupper():
            return target_city.title()
        return target_city.lower()

    return CITY_PATTERN.sub(repl, text)

english_eval = df[df["has_city_mention"]].copy()
english_eval["source_city"] = english_eval["city_mentions"].apply(lambda xs: xs[0] if xs else None)
english_eval = english_eval[english_eval["source_city"].notna()].copy()

counterfactual_rows = []
for _, row in english_eval.iterrows():
    for swap_city in SWAP_CITIES:
        if row["source_city"] == swap_city:
            continue
        swapped_text = swap_cities_in_text(row["resume_text_work"], swap_city)
        counterfactual_rows.append(
            {
                "row_id": int(row.name),
                "target_9_class": row["target_9_class"],
                "source_city": row["source_city"],
                "swap_city": swap_city,
                "original_text": row["resume_text_work"],
                "swapped_text": swapped_text,
            }
        )

counterfactual_df = pd.DataFrame(counterfactual_rows)
base_eval_df = english_eval[["resume_text_work", "target_9_class", "source_city"]].rename(columns={"resume_text_work": "original_text"})

base_eval_df.to_csv(OUTPUT_DIR / "02_english_base_eval_slice.csv", index=False)
counterfactual_df.to_csv(OUTPUT_DIR / "02_english_city_swap_counterfactuals.csv", index=False)

print("Base eval rows:", len(base_eval_df))
print("Counterfactual rows:", len(counterfactual_df))
counterfactual_df.head(10)


In [ ]:
MODEL_CANDIDATES = [
    ("baseline", REPO_ROOT / "models" / "bert_9classes_final"),
    ("groupdro", REPO_ROOT / "models" / "bert_gdro_eta01_2ep"),
    ("label_smoothing", REPO_ROOT / "models" / "bert_label_smoothing"),
    ("combined_best", REPO_ROOT / "models" / "combined_scrubbing_groupdro" / "final"),
]

available_models = [(name, path) for name, path in MODEL_CANDIDATES if path.exists()]
if not available_models:
    print("No local checkpoints found under repo/models. Dataset preparation is complete; run evaluation on Natasha's machine where checkpoints exist.")
else:
    print("Found local checkpoints:")
    for name, path in available_models:
        print(" -", name, "->", path)


In [ ]:
evaluation_plan = pd.DataFrame(
    [
        {
            "priority": 1,
            "step": "Manual review of English->9-class mapping",
            "why": "Prevents garbage transfer metrics from label mismatch",
            "artifact": "02_english_category_mapping_review.csv",
        },
        {
            "priority": 2,
            "step": "Run existing strongest models on the English base eval slice",
            "why": "Checks out-of-domain generalization before any retraining",
            "artifact": "02_english_base_eval_slice.csv",
        },
        {
            "priority": 3,
            "step": "Run city-swap on the English counterfactual slice",
            "why": "Measures location sensitivity on English resumes directly",
            "artifact": "02_english_city_swap_counterfactuals.csv",
        },
        {
            "priority": 4,
            "step": "Only then decide whether to fine-tune challengers on English data",
            "why": "Avoids another 15 training runs if transfer already looks acceptable",
            "artifact": "decision_after_eval",
        },
    ]
)
evaluation_plan.to_csv(OUTPUT_DIR / "02_english_next_steps_plan.csv", index=False)
evaluation_plan


In [ ]:
quick_answer = {
    "should_we_do_city_swap_on_english": True,
    "should_we_train_again_immediately": False,
    "recommended_first_move": "map English labels into the current 9-class space and evaluate existing strongest checkpoints first",
    "why": [
        "The dataset must be aligned to the current label space before city-swap numbers are interpretable.",
        "Existing models may already transfer well enough to justify evaluation-before-retraining.",
        "If transfer fails, the prepared English counterfactual slice becomes the benchmark for the next training wave.",
    ],
}
(OUTPUT_DIR / "02_english_quick_answer.json").write_text(
    json.dumps(quick_answer, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(json.dumps(quick_answer, indent=2, ensure_ascii=False))


In [ ]:
if HAS_MPL and not df.empty:
    class_counts = df["target_9_class"].value_counts().sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    class_counts.plot(kind="barh", ax=ax, color="#4c78a8")
    ax.set_title("English dataset mapped into current 9-class space")
    ax.set_xlabel("Rows")
    ax.set_ylabel("Target class")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "02_english_target9_distribution.png", dpi=180, bbox_inches="tight")
    plt.show()
